In [1]:
import sys
import socket
import importlib
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
cd ../../work/nvme/bhvr/cadence/Stars_VAE

/work/nvme/bhvr/cadence/Stars_VAE


In [7]:
ls

data/  environment.yml  notebooks/  README.md  requirements.txt  src/


In [9]:
from src.config import raw_data_dir

data_path = raw_data_dir / "sample_data.npy"
sim = np.load(data_path, allow_pickle=True)

df = pd.DataFrame(sim)

In [11]:
# importlib.reload(sys.modules["src.preprocessing"])
from src.preprocessing import check_missing_values, drop_constant_columns, fit_preprocess_scalers



ImportError: cannot import name 'fit_preprocess_scalers' from partially initialized module 'src.preprocessing' (most likely due to a circular import) (/work/nvme/bhvr/cadence/Stars_VAE/src/preprocessing.py)

In [ ]:
df_rdp = rdp(df, epsilon=rdp_epsilon)
processed_data_path.parent.mkdir(parents=True, exist_ok=True)
np.save(processed_data_path, df_rdp.to_numpy())
print(f"Saved processed dataset to {processed_data_path}")

In [ ]:
df.head(20)

In [ ]:
check_missing_values(df_rdp)
print(f"Data shadata/processedpe after RDP vs original shape: {df_rdp.shape} vs {df.shape}")

df_reduced = drop_constant_columns(df_rdp)

df_subset = df_reduced[target_features]

df_scaled, scalers = fit_preprocess_scalers(
    df_subset,
    normalize=True,
    standardize=False,
)


In [ ]:
# Isolate relevant features
df_visual = df[target_features + ['star_age']]

# Construct the 3D structure using groupby for computational efficiency
stratified_data = {
    age: group[target_features] 
    for age, group in df_visual.groupby('star_age')
}

# Extract the first available age to verify structure
first_age = list(stratified_data.keys())[0]

max_age_slice = max(stratified_data, key=lambda age: len(stratified_data[age]))
max_datapoints = len(stratified_data[max_age_slice])
min_age_slice = min(stratified_data, key=lambda age: len(stratified_data[age]))
min_datapoints = len(stratified_data[min_age_slice])

num_age_slices = len(stratified_data)


print(f"Time slice: {min_age_slice}")
print(f"Datapoint count: {min_datapoints}")
print(f"Time slice: {max_age_slice}")
print(f"Datapoint count: {max_datapoints}")
print(f"Total number of age slices: {num_age_slices}")

In [ ]:
import numpy as np
import pandas as pd
from src.config import num_profile_points, split_feature, profile_features
from src.preprocessing import fit_preprocess_scalers, process_simulation

def rdp_preprocess(raw_df, split_feature=split_feature, profile_features=profile_features, num_profile_points=num_profile_points):
# Normalized minmax scaled (between 0 & 1)
	normalized_df = fit_preprocess_scalers(raw_df, profile_features + [split_feature], True, False)
# stratify data into profiles based on a split feature
	profiles, splitting_features = process_simulation(normalized_df)
	#ensure the indicies from the original raw_df is preserved in the new profiles and their points

	# determine which profile has the minimum number of points
	min_points = profiles.argmin #replace with real code

# ensure inputed num profile points is acceptible 
	if num_profile_points > min_points:
		num_profile_points = min_points

	for profile, feature in zip(profiles, splitting_features):
		while len(profiles) < num_profile_points:
			# implement rdp algorithm with 2 variations. One is that we will use max heap instead of epsilon to ensure that the num of points threashold is met. 2 is that it is itterative, not recursive. Implement a perpendicular distance from line function too to help the rdp. The algorithm should return a 1d array containg the indecies of points in each profile and the splitting feature value of those profiles
			pass
	# create a to numpy array that contains the data from raw_df only selecting the columns profile_features + [split_feature] and only selecting the rows that have the indecies that the above rdp algorithm found. The splitting feature should be constant not empty in this returned npy dataframe for the changing profile features. 
	# return this numpy array 

In [ ]:
cd ../../work/nvme/bhvr/cadence/Stars_VAE

In [ ]:
import unittest
import numpy as np
import pandas as pd
from unittest.mock import patch

# Adjust import statement based on your file naming convention.
# Assuming the algorithm is stored in 'rdp_algorithm.py'
from src.preprocessing import get_max_perpendicular_distance, iterative_rdp_max_heap, rdp_preprocess

class TestRDPAlgorithm(unittest.TestCase):
    
    def test_get_max_perpendicular_distance(self):
        points = np.array([
            [0.0, 0.0],
            [1.0, 2.0],  # Major deviation
            [2.0, 0.5],
            [3.0, 0.0]
        ])
        
        dist, split_idx = get_max_perpendicular_distance(points, 0, 3)
        self.assertAlmostEqual(dist, 1.89736659, places=5)
        self.assertEqual(split_idx, 1)

    def test_get_max_perpendicular_distance_collinear(self):
        points = np.array([
            [0.0, 0.0],
            [1.0, 1.0],
            [2.0, 2.0],
            [3.0, 3.0]
        ])
        
        dist, split_idx = get_max_perpendicular_distance(points, 0, 3)
        self.assertAlmostEqual(dist, 0.0)

    def test_iterative_rdp_max_heap(self):
        points = np.array([
            [0.0, 0.0],
            [1.0, 0.1],  # Minor noise
            [2.0, 5.0],  # Major peak
            [3.0, 0.2],  # Minor noise
            [4.0, 0.0]
        ])
        original_indices = np.array([100, 101, 102, 103, 104])
        
        target_points = 3
        rdp_indices = iterative_rdp_max_heap(points, original_indices, target_points)
        
        # Retains start (100), end (104), and the maximum deviation point (102)
        np.testing.assert_array_equal(rdp_indices, [100, 102, 104])

    def test_iterative_rdp_target_exceeds_points(self):
        points = np.array([[0,0], [1,1], [2,0]])
        original_indices = np.array([0, 1, 2])
        rdp_indices = iterative_rdp_max_heap(points, original_indices, 10)
        
        # Returns original indices unaltered
        np.testing.assert_array_equal(rdp_indices, [0, 1, 2])

    @patch('rdp_algorithm.process_simulation')
    @patch('rdp_algorithm.fit_preprocess_scalers')
    def test_rdp_preprocess_integration(self, mock_fit, mock_process):
        np.random.seed(42)
        raw_df = pd.DataFrame({
            'feature1': np.random.rand(10),
            'feature2': np.random.rand(10),
            'zone': [1.0]*5 + [2.0]*5
        }, index=[10, 11, 12, 13, 14, 20, 21, 22, 23, 24])

        mock_fit.return_value = raw_df 
        
        profile_1 = raw_df.iloc[0:5]
        profile_2 = raw_df.iloc[5:10]
        mock_process.return_value = ([profile_1, profile_2], [1.0, 2.0])

        profile_features = ['feature1', 'feature2']
        split_feature = 'zone'
        target_points = 3
        
        output_npy = rdp_preprocess(
            raw_df=raw_df,
            split_feature=split_feature,
            profile_features=profile_features,
            num_profile_points=target_points
        )
        
        # Verification: Output matrix dimensions (2 profiles * 3 points = 6 rows, 3 columns)
        self.assertEqual(output_npy.shape, (6, 3))
        
        # Verification: Correct column extraction based on dummy data
        self.assertTrue(np.all(output_npy[:, 2] == np.array([1.0, 1.0, 1.0, 2.0, 2.0, 2.0])))

if __name__ == '__main__':
    unittest.main()

In [ ]:
# from scipy.interpolate import interp1d
# import numpy as np
# import pandas as pd

# def perpendicular_distance(points, start, end):
#     """
#     Computes the perpendicular distance from an array of N-dimensional points 
#     to the line defined by start and end points.
#     """
#     # Handle case where start and end points are identical
#     if np.allclose(start, end):
#         return np.linalg.norm(points - start, axis=1)
        
#     line_vec = end - start
#     point_vec = points - start
    
#     # Unit vector of the line segment
#     line_unitvec = line_vec / np.linalg.norm(line_vec)
    
#     # Project point vectors onto the line unit vector
#     proj_lengths = np.dot(point_vec, line_unitvec)
#     proj_vecs = np.outer(proj_lengths, line_unitvec)
    
#     # Distance is the norm of the difference between original vector and projected vector
#     return np.linalg.norm(point_vec - proj_vecs, axis=1)

# def rdp_custom(points, epsilon):
#     """
#     Native implementation of the Ramer-Douglas-Peucker algorithm.
#     """
#     # Base case: a line segment of 2 points cannot be simplified further
#     if len(points) <= 2:
#         return points

#     # Compute distances of all interior points to the line connecting start and end
#     distances = perpendicular_distance(points[1:-1], points[0], points[-1])
    
#     # Find the point with the maximum distance
#     index = np.argmax(distances) + 1 
#     dmax = distances[index - 1]

#     # If maximum distance exceeds epsilon, recursively simplify both halves
#     if dmax > epsilon:
#         left = rdp_custom(points[:index+1], epsilon)
#         right = rdp_custom(points[index:], epsilon)
        
#         # Stack halves, dropping the duplicated vertex at the split point
#         return np.vstack((left[:-1], right))
#     else:
#         # All interior points are within epsilon; discard them
#         return np.array([points[0], points[-1]])

# def rdp_exact_count(points, target_points, tolerance=1e-6):
#     """
#     Executes a binary search across the epsilon parameter space to force RDP 
#     to output an exact number of vertices.
#     """
#     if len(points) <= target_points:
#         return points
    
#     eps_low = 0.0
#     eps_high = np.max(np.linalg.norm(points - points[0], axis=1))

#     best_points = points
    
#     while (eps_high - eps_low) > tolerance:
#         eps_mid = (eps_low + eps_high) / 2.0
#         simplified = rdp_custom(points, epsilon=eps_mid)
#         count = len(simplified)

#         if count == target_points:
#             return simplified
#         elif count > target_points:
#             eps_low = eps_mid
#             # Restrict best_points to the smallest subset that is still >= target
#             if count < len(best_points):
#                 best_points = simplified
#         else:
#             eps_high = eps_mid

#     return best_points

# def process_simulation(sim, rdp_feature):
#     """ This function processes a simulation into 
#         a list of profiles.

#         Parameter(s):
#             sim - a loaded numpy structured array which has all of the zones
#                   of a simulation for a spatio-temporal evolution as rows

#         Return Value(s):
#             ages - a list of ages corresponding to each profile
#             profiles - the profiles
#     """
#     zone_starts = np.where(sim["zone"] == 1.0)[0] # finding all zones at interior or exterior of star (depends on setup)

#     profiles = []
#     ages = []

#     for i, start in enumerate(zone_starts):
#         if i < len(zone_starts) - 1:
#             profiles.append(
#                 sim[start:zone_starts[i + 1]] # getting zones belonging to profile
#             )

#         else:
#             profiles.append(
#                 sim[start:] # edge case for last zone
#             )

#         ages.append(
#             sim[start][rdp_feature] # collecting ages
#         )

#     ages = np.array(ages)

#     # ordering profiles by age
#     ordered_inds = ages.argsort()
#     profiles = np.array(profiles, dtype = object)
#     profiles = profiles[ordered_inds]
#     ages = ages[ordered_inds]

#     return ages, profiles

# def apply_rdp_to_slices(sim, rdp_feature):
#     """
#     Main execution pipeline. Stratifies data, determines target count, 
#     and applies forced-count RDP to all slices.
#     """
#     ages, profiles = process_simulation(sim, rdp_feature)
#     processed_slices = {}
    
#     # Determine target count from the processed array of slices
#     min_datapoints = min(len(p) for p in profiles)
    
#     for age, profile in zip(ages, profiles):
#         # Convert structured numpy array to unstructured 2D float array
#         points = np.column_stack([profile[name] for name in profile.dtype.names])
        
#         simplified_points = rdp_exact_count(points, target_points=min_datapoints)
        
#         if len(simplified_points) > min_datapoints:
#             indices = np.round(np.linspace(0, len(simplified_points) - 1, min_datapoints)).astype(int)
#             simplified_points = simplified_points[indices]
            
#         processed_slices[age] = pd.DataFrame(simplified_points, columns=profile.dtype.names)
        
#     return processed_slices

In [ ]:
df_visual.head()

In [ ]:
rdp_feature = 'star_age'

processed_df = apply_rdp_to_slices(df_visual, rdp_feature)

In [ ]:
stratified_data = rdp(df_visual, .05, rdp_feature)

first_age = list(stratified_data.keys())[0]

max_age_slice = max(stratified_data, key=lambda feature: len(stratified_data[feature]))
max_datapoints = len(stratified_data[max_age_slice])
min_age_slice = min(stratified_data, key=lambda feature: len(stratified_data[feature]))
min_datapoints = len(stratified_data[min_age_slice])

num_age_slices = len(stratified_data)


print(f"Time slice: {min_age_slice}")
print(f"Datapoint count: {min_datapoints}")
print(f"Time slice: {max_age_slice}")
print(f"Datapoint count: {max_datapoints}")
print(f"Total number of age slices: {num_age_slices}")

In [ ]:
# importlib.reload(sys.modules["src.train"])
from src.train import train_model

In [ ]:
metrics = train_model()

In [ ]:
epochs = range(len(metrics["train_mse"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(epochs, metrics["train_mse"], label="Train MSE")
ax1.plot(epochs, metrics["val_mse"], label="Validation MSE")
ax1.set_title("Reconstruction Loss (MSE)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(epochs, metrics["train_kld"], label="Train KLD")
ax2.plot(epochs, metrics["val_kld"], label="Validation KLD")
ax2.set_title("Latent Divergence (KLD)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("KLD")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()